# local_linear

> 对应代码：`EconometricsCode/code_in_notes/Chap.3/local_linear.do`（本 notebook 中的 Stata 代码与 do 文件一致）
>
> 运行方式：通过 *PyStata* 在 Jupyter 中调用 Stata，安装与使用说明见 `notebooks/PyStata.md`。

**说明**：为在 notebook 中直接内联显示图形，本 notebook 的 Stata 代码省略了 `.do` 文件中的 `graph export` 导出命令（`code_in_notes` 中对应的 .do 文件保留导出，用于生成书中的插图）。

首先启动 Stata 环境，并把工作目录切换到 `code_in_notes/Chap.3`，这样 Stata 中的相对路径（如 `../datasets/...`）与同名 do 文件完全一致，图片也会输出到该目录。

In [1]:
# 配置 PyStata：启动 Stata 并注册 %%stata 魔法命令
import os
os.chdir("../../code_in_notes/Chap.3")

import stata_setup
stata_setup.config("/opt/Stata", "mp")


  ___  ____  ____  ____  ____ ®
 /__    /   ____/   /   ____/      Stata 18.0
___/   /   /___/   /   /___/       MP—Parallel Edition

 Statistics and Data Science       Copyright 1985-2023 StataCorp LLC
                                   StataCorp
                                   4905 Lakeway Drive
                                   College Station, Texas 77845 USA
                                   800-782-8272        https://www.stata.com
                                   979-696-4600        service@stata.com

Stata license: Single-user 16-core  perpetual
Serial number: 501806307019
  Licensed to: Keep calm and DO your research
               Jichun Si

Notes:
      1. Unicode is supported; see help unicode_advice.
      2. More than 2 billion observations are allowed; see help obs_advice.
      3. Maximum number of variables is set to 5,000 but can be increased;
          see help set_maxvar.


文件开始，我们先清空内存并关闭分屏显示，让后续命令在干净的环境中连续执行。

In [2]:
%%stata
clear
set more off


. clear

. set more off

. 


我们生成模拟数据：解释变量 x 服从 (0,2) 上的均匀分布，真实回归函数为 m(x)=exp(sin(x³))，这是一个非线性、曲率变化丰富的函数；观测值 y 在 m(x) 上叠加标准正态噪声（设定种子保证结果可复现）。我们不预设函数形式，目标是仅用数据在 x=2 附近局部地估计 m(2)。

In [3]:
%%stata
// 生成数据
set seed 19880505
set obs 300
gen x=2*runiform()
gen y=exp(sin(x^3))+rnormal()


. // 生成数据
. set seed 19880505

. set obs 300
Number of observations (_N) was 0, now 300.

. gen x=2*runiform()

. gen y=exp(sin(x^3))+rnormal()

. 


接下来构造核权重。局部回归的思想是：要估计 m(x) 在 x₀=2 处的取值，只有 x 靠近 2 的观测才包含有效信息，离得远的点应被压低权重。normalden((x-2)/0.1) 是以 2 为中心、带宽 h=0.1 的正态核密度，x 越接近 2 权重越大，距离超过两三个带宽后权重几乎为零。带宽 h 控制“局部”的范围：h 越小越局部，偏差小但方差大。

In [4]:
%%stata
// 局部线性回归
gen w=normalden((x-2)/0.1)


. // 局部线性回归
. gen w=normalden((x-2)/0.1)

. 


我们把解释变量平移为 x-2，即中心化到估计点 x₀=2。这样在 x₀ 处 x-2=0，回归常数项正好对应 m(2) 的估计，斜率系数对应导数 m′(2) 的估计。

In [5]:
%%stata
gen x_2=x-2

核心估计命令：以 w 为重要性权重（[iw=w]）做 y 对 x_2 的加权最小二乘回归，等价于在 x₀=2 处最小化加权残差平方和 Σᵢ wᵢ(yᵢ−β₀−β₁(xᵢ−2))²。这就是局部线性回归（local linear regression）：在估计点附近用一条直线逼近真实函数，β̂₀ 给出 m(2)，β̂₁ 给出 m′(2)。相比局部常数（Nadaraya–Watson）估计，局部线性在边界附近偏差更小。

In [6]:
%%stata
reg y x_2 [iw=w]


      Source |       SS           df       MS      Number of obs   =         8
-------------+----------------------------------   F(1, 6)         =      2.16
       Model |  2.67752786         1  2.67752786   Prob > F        =    0.1917
    Residual |  7.62757769         6  1.27126295   R-squared       =    0.2598
-------------+----------------------------------   Adj R-squared   =    0.1594
       Total |  10.3051056         7  1.47215794   Root MSE        =    1.1124

------------------------------------------------------------------------------
           y | Coefficient  Std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
         x_2 |   10.13351   6.889007     1.47   0.192    -6.723278    26.99031
       _cons |   2.776023   .6180733     4.49   0.004     1.263652    4.288394
------------------------------------------------------------------------------


显示常数项估计，即 m̂(2)。真实值为 m(2)=exp(sin(8))≈2.69，我们可以对照检查估计的精度。

In [7]:
%%stata
di _b[_cons]

2.7760227
